<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B01%5D%20-%20Notebooks/E3_KMedoids_E2E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · K-Medoids E2E - Introducción a los modelos no supervisados

## Introducción

El **centroide** de K-Means es un punto medio inventado: normalmente no existe ningún cliente
justo ahí. ¿Y si quieres un **cliente real** que represente al grupo? Para eso está
**K-Medoids**: usa como centro un **dato real**, la observación más representativa del grupo (el
**medoide**).

En este ejercicio, de principio a fin (end-to-end):

1. Reutilizamos el **K** elegido en K-Means (E2).
2. Implementamos **K-Medoids desde cero** y lo aplicamos.
3. Comparamos **centroide (promedio) vs medoide (real)**.
4. Identificamos el **cliente representante** de cada grupo.
5. Describimos **en una frase** qué parece representar cada cluster.

## Objetivos del ejercicio

- Entender la diferencia entre **centroide** (promedio) y **medoide** (dato real).
- Implementar y aplicar **K-Medoids** con el mismo K que K-Means.
- Identificar el **cliente representante** (medoide) de cada grupo.
- Ver por qué K-Medoids es **más robusto a outliers**.

## Descripción del dataset (clientes sin etiqueta)

Usamos el mismo dataset **sintético y reproducible** de clientes que en E1 y E2 (creado con
`generar_clientes`, autocontenido en Colab). Cada fila es un cliente con variables de negocio
(`gasto_anual`, `num_visitas`, `ticket_medio`, `antiguedad_meses`, `edad`) y varias binarias
(`usa_app`, `tiene_tarjeta_fidelidad`, `compra_online`...). Las escalas son muy distintas, así
que **escalamos antes de medir distancias**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 120)

### 2. Datos, escalado y el K elegido en K-Means

In [ ]:
import numpy as np
import pandas as pd

def generar_clientes(n=600, semilla=42):
    # Dataset sintetico y reproducible de clientes SIN ETIQUETA para segmentacion.
    # Por dentro hay 4 perfiles latentes que el modelo deberia redescubrir, pero NO los
    # exponemos: en aprendizaje no supervisado no hay target, solo buscamos estructura.
    rng = np.random.default_rng(semilla)
    # perfil: (gasto_anual, num_visitas, ticket_medio, antiguedad_meses, edad,
    #          p_app, p_fidelidad, p_online, p_newsletter, p_devuelve)
    perfiles = [
        (9000, 42, 230, 60, 46, 0.85, 0.90, 0.70, 0.60, 0.10),  # grandes clientes
        (1100,  6, 120, 22, 37, 0.40, 0.20, 0.55, 0.30, 0.10),  # ocasionales
        (3200, 36,  75, 44, 52, 0.50, 0.65, 0.60, 0.80, 0.55),  # cazaofertas
        (2400, 15, 165,  9, 30, 0.92, 0.40, 0.95, 0.50, 0.20),  # nuevos digitales
    ]
    pesos = [0.22, 0.33, 0.25, 0.20]
    seg = rng.choice(len(perfiles), size=n, p=pesos)

    filas = []
    for s in seg:
        g, v, t, a, e, pa, pf, po, pn, pdv = perfiles[s]
        filas.append([
            round(max(50, rng.normal(g, g * 0.22)), 2),    # gasto_anual (€)
            int(max(1, round(rng.normal(v, v * 0.30)))),    # num_visitas
            round(max(5, rng.normal(t, t * 0.22)), 2),      # ticket_medio (€)
            int(max(1, round(rng.normal(a, 12)))),          # antiguedad_meses
            int(np.clip(rng.normal(e, 8), 18, 85)),         # edad
            int(rng.random() < pa),                          # usa_app
            int(rng.random() < pf),                          # tiene_tarjeta_fidelidad
            int(rng.random() < po),                          # compra_online
            int(rng.random() < pn),                          # recibe_newsletter
            int(rng.random() < pdv),                         # devuelve_productos
        ])
    cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad",
            "usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter",
            "devuelve_productos"]
    return pd.DataFrame(filas, columns=cols)

In [ ]:
df = generar_clientes(n=600, semilla=42)
X = df.to_numpy(dtype=float)
X_esc = StandardScaler().fit_transform(X)

# Reutilizamos el mismo K que en E2 (el que maximiza el silhouette)
mejor_k, mejor_sil = None, -1
for k in range(2, 7):
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    s = silhouette_score(X_esc, lab)
    if s > mejor_sil:
        mejor_k, mejor_sil = k, s

km = KMeans(n_clusters=mejor_k, init="k-means++", n_init=10, random_state=0).fit(X_esc)
print(f"K elegido en K-Means: {mejor_k} (silhouette {mejor_sil:.3f})")

### 3. K-Medoids desde cero

El algoritmo es muy parecido al de K-Means, pero el centro es siempre un punto real:

1. Elige K medoides iniciales (K clientes).
2. Asigna cada punto a su **medoide más cercano**.
3. En cada grupo, el nuevo medoide es el **cliente que minimiza la distancia total** al resto
   de su grupo.
4. Repite 2-3 hasta que los medoides no cambian.

> Nota: existe `KMedoids` en la librería `scikit-learn-extra`, pero aquí lo implementamos a mano
> (son pocas líneas) para entenderlo y no depender de instalaciones extra.

In [ ]:
def kmedoids(X, k, semilla=0, max_iter=100):
    rng = np.random.default_rng(semilla)
    D = squareform(pdist(X))                 # matriz de distancias n x n (euclídea)
    n = X.shape[0]
    medoides = rng.choice(n, size=k, replace=False)   # K clientes iniciales
    for _ in range(max_iter):
        labels = D[:, medoides].argmin(axis=1)        # asignar al medoide más cercano
        nuevos = medoides.copy()
        for c in range(k):
            idx = np.where(labels == c)[0]
            if len(idx) == 0:
                continue
            # nuevo medoide = el punto del grupo con menor distancia total a su grupo
            nuevos[c] = idx[D[np.ix_(idx, idx)].sum(axis=1).argmin()]
        if set(nuevos) == set(medoides):              # convergencia
            break
        medoides = nuevos
    labels = D[:, medoides].argmin(axis=1)
    return medoides, labels

medoides_idx, labels_med = kmedoids(X_esc, mejor_k, semilla=0)
print("Índices de los clientes-medoide:", medoides_idx)
print("Tamaño de cada cluster (K-Medoids):", np.bincount(labels_med))
print(f"\nSilhouette K-Means:   {silhouette_score(X_esc, km.labels_):.3f}")
print(f"Silhouette K-Medoids: {silhouette_score(X_esc, labels_med):.3f}")

### 4. Centroide (promedio) vs Medoide (real)

El centroide de K-Means es un promedio (no es un cliente). El medoide de K-Medoids **es una
fila real** del dataset. Veámoslos en sus unidades originales:

In [ ]:
cols_clave = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad"]

centroides = StandardScaler().fit(X).inverse_transform(km.cluster_centers_)
print("Centroides de K-Means (PROMEDIO, no son clientes reales):")
print(pd.DataFrame(centroides, columns=df.columns)[cols_clave].round(1).to_string())

print("\nMedoides de K-Medoids (CLIENTES REALES, filas del dataset):")
print(df.iloc[medoides_idx][cols_clave].round(1).to_string())

### 5. El cliente representante de cada grupo

In [ ]:
representantes = df.iloc[medoides_idx].copy()
representantes.insert(0, "id_cliente", medoides_idx)
representantes.insert(1, "cluster", range(mejor_k))
representantes.set_index("cluster")

### 6. Describe cada cluster en una frase

Con el medoide (cliente real) de cada grupo delante, lo resumimos en una frase comparando sus
valores con la **mediana** de todos los clientes (referencia de "típico").

In [ ]:
ref = df[cols_clave].median().to_numpy()
vals = df.iloc[medoides_idx][cols_clave].to_numpy()

for c in range(mejor_k):
    v = vals[c]
    rasgos = [
        "gasto " + ("alto" if v[0] > ref[0] else "bajo"),
        ("muchas" if v[1] > ref[1] else "pocas") + " visitas",
        "ticket " + ("alto" if v[2] > ref[2] else "bajo"),
        ("cliente veterano" if v[3] > ref[3] else "cliente reciente"),
    ]
    print(f"Cluster {c}: " + ", ".join(rasgos) + ".")

### 7. ¿Por qué K-Medoids es más robusto a outliers?

El centroide es un **promedio**, y los promedios se disparan con un valor extremo. El medoide,
al ser un punto real central (parecido a la **mediana**), aguanta mucho mejor. Veámoslo con el
gasto del cluster 0:

In [ ]:
g = df.loc[labels_med == 0, "gasto_anual"].to_numpy()
g_out = np.append(g, 500000.0)     # entra un cliente atípico (gasto enorme)

print(f"Gasto medio del grupo (tipo centroide): {g.mean():9.0f}")
print(f"Gasto del medoide (cliente real):       {df.iloc[medoides_idx[0]]['gasto_anual']:9.0f}")
print("\nSi entra UN cliente atípico (gasto 500000 €):")
print(f"  el PROMEDIO se dispara:            {g.mean():.0f} -> {g_out.mean():.0f}")
print(f"  la MEDIANA (idea del medoide) casi no cambia: {np.median(g):.0f} -> {np.median(g_out):.0f}")
print("\nPor eso K-Medoids es más robusto al ruido, aunque es más lento que K-Means.")

### Reflexión

1. ¿En qué se diferencia un **centroide** de un **medoide**?
2. Mira el cliente representante de cada grupo: ¿te parece coherente con el resto del grupo?
3. ¿Por qué el medoide aguanta mejor un outlier que el centroide?
4. ¿Cuándo preferirías K-Medoids frente a K-Means? ¿Y al revés?